# LLM-as-a-Judge Evaluator for Agent Interventions

This notebook is used for the purpose of developing and validating the judge LLM. 

For each intervention, the judge is shown:
- The document and highlighted sentence
- The selected comment thread (based on `comments_used` field)
- The agent interventions

The judge can rate the agent intervention or select which one is preferred.

In [1]:
from src import models, llms, judge, utils

# Initialize LLM client with gpt-4o model and lower temperature for consistent judging
judge_llm = llms.OpenAiClient(model="gpt-4.1-mini", temperature=0.3)


## Run judge as a rater

In [2]:
print(judge.judge_rate_intervention(
    llm=judge_llm,
    document="This is a generic document about something uninteresting.\nHere's another paragraph.\nAnd here's another one that is somewhat controversial.",
    highlighted_sentence="And here's another one that is somewhat controversial.",
    comment_thread=[
        {
            "speaker": "One",
            "text": "I don't think this is controversial."
        },
        {
            "speaker": "Two",
            "text": "I do"
        },
        {
            "speaker": "One",
            "text": "This is a stalemate!"
        }
    ],
    intervention="NO_INTERVENTION",
    comments_used=3,
    agent_uses_chain_of_thought=False,
    retries=0
))

{'reasoning': "The discussion has clearly reached a stalemate with both users reiterating their opposing views without progress. The assistant's choice to not intervene misses an opportunity to break the deadlock by prompting clarification or suggesting a way to reconcile the disagreement. Therefore, the lack of intervention at this point is unhelpful and unlikely to move the discussion forward.", 'rating': 1}


## Run Judge as a Selector

In [2]:
print(judge.judge_select_intervention(
    llm=judge_llm,
    document="This is a generic document about something uninteresting.\nHere's another paragraph.\nAnd here's another one that is somewhat controversial.",
    highlighted_sentence="And here's another one that is somewhat controversial.",
    comment_thread=[
        {
            "speaker": "One",
            "text": "I don't think this is controversial."
        },
        {
            "speaker": "Two",
            "text": "I do"
        },
        {
            "speaker": "One",
            "text": "This is a stalemate!"
        }
    ],
    intervention_1="NO_INTERVENTION",
    intervention_2="This is completely unhelpful comment and won't help resolve the conflict.",
    comments_used=3,
    agent_uses_chain_of_thought=False,
    retries=1
))

{'reasoning': 'Intervention 1 appropriately refrains from intervening prematurely, allowing the users to continue their discussion naturally. Intervention 2 is formatted as a direct critique of a user comment rather than a constructive contribution to the discussion, which is unhelpful and could escalate conflict rather than resolve it.', 'selection': 1}
